In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils, paths
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"
# Which result pathways this (location, vehicle) combo has. Snakemake passes
# these from config_utils.get_combo_pathways, the same function that decides
# which inputs the rule declares. A pathway the combo lacks contributes zeros,
# built from the template combo (india/rice). Never branch on whether a file
# exists: a job can start before the file is built, and a file missing by
# mistake would silently become zeros.
has_simulations = True
has_anemia_model = True
has_ntd_model = True
data_source = paths.data_source(default="local")  # or a model number on the team drive

In [3]:
# Parameters
location = "india"
vehicle = "wheat"
has_simulations = True
has_anemia_model = True
has_ntd_model = True


In [4]:
rescaled_pregnancy_root = paths.source_root(paths.RESCALED_PREGNANCY_RESULTS_ROOT, data_source)
rescaled_child_root = paths.source_root(paths.RESCALED_CHILD_RESULTS_ROOT, data_source)
anemia_root = paths.source_root(paths.NON_PREGNANT_ANEMIA_RESULTS_ROOT, data_source)
print(f"{data_source}:\n  {rescaled_pregnancy_root}\n  {rescaled_child_root}\n  {anemia_root}")

model2.6.4:
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.4/rescaled_pregnancy
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.4/rescaled_child
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.4/non_pregnant_anemia


In [5]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [6]:
path = f"{rescaled_pregnancy_root}/{vehicle}/{location}/person_time_anemia.parquet"
if has_simulations:
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"{rescaled_pregnancy_root}/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,141,13.687242
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,141,4.700669
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,141,7.327513
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,141,10.645632
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,141,8.387468
...,...,...,...,...,...,...,...,...,...,...,...
1079995,person_time,impairment,anemia,severe,95_plus,severe,1,zero,0,129,0.000000
1079996,person_time,impairment,anemia,severe,95_plus,severe,2,zero,0,129,0.000000
1079997,person_time,impairment,anemia,severe,95_plus,severe,3,zero,0,129,0.000000
1079998,person_time,impairment,anemia,severe,95_plus,severe,4,zero,0,129,0.000000


In [7]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [8]:
pregnancy_person_time_anemia.sub_entity.value_counts()

sub_entity
not_anemic    270000
mild          270000
moderate      270000
severe        270000
Name: count, dtype: int64

In [9]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  4.931124e+06
              2                  4.078587e+06
              3                  3.655052e+06
              4                  3.475273e+06
              5                  3.393210e+06
intervention  1                  4.931127e+06
              2                  4.078587e+06
              3                  3.655052e+06
              4                  3.475276e+06
              5                  3.393211e+06
zero          1                  4.931124e+06
              2                  4.078587e+06
              3                  3.655052e+06
              4                  3.475273e+06
              5                  3.393210e+06
Name: value, dtype: float64

In [10]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.825995e+06
              2                  2.221178e+06
              3                  1.891708e+06
              4                  1.664798e+06
              5                  1.376576e+06
intervention  1                  2.807312e+06
              2                  2.205702e+06
              3                  1.877309e+06
              4                  1.651109e+06
              5                  1.361751e+06
zero          1                  2.827219e+06
              2                  2.222241e+06
              3                  1.892720e+06
              4                  1.665696e+06
              5                  1.377492e+06
Name: value, dtype: float64

In [11]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.573094
              2                  0.544595
              3                  0.517560
              4                  0.479041
              5                  0.405686
intervention  1                  0.569304
              2                  0.540801
              3                  0.513620
              4                  0.475102
              5                  0.401316
zero          1                  0.573342
              2                  0.544856
              3                  0.517837
              4                  0.479299
              5                  0.405955
Name: value, dtype: float64

In [12]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [13]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,48678.711021
1,Female,0.0,0.019178,not_pregnant,2,42723.422230
2,Female,0.0,0.019178,not_pregnant,3,37932.716572
3,Female,0.0,0.019178,not_pregnant,4,35728.167677
4,Female,0.0,0.019178,not_pregnant,5,28631.476092
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,18328.880607
281,Male,95.0,125.000000,not_pregnant,2,19140.234171
282,Male,95.0,125.000000,not_pregnant,3,19770.250303
283,Male,95.0,125.000000,not_pregnant,4,20851.307187


In [14]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    4.478246e+06
2    3.707329e+06
3    3.317141e+06
4    3.157155e+06
5    3.080667e+06
Name: value, dtype: float64

In [15]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.566454e+06
              2                  2.018993e+06
              3                  1.716819e+06
              4                  1.512407e+06
              5                  1.249782e+06
intervention  1                  2.549485e+06
              2                  2.004926e+06
              3                  1.703751e+06
              4                  1.499969e+06
              5                  1.236322e+06
zero          1                  2.567566e+06
              2                  2.019959e+06
              3                  1.717737e+06
              4                  1.513222e+06
              5                  1.250613e+06
Name: value, dtype: float64

In [16]:
path = f"{rescaled_pregnancy_root}/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if has_simulations:
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"{rescaled_pregnancy_root}/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,141,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,141,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,141,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,141,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,zero,0,129,0.0
539996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,zero,0,129,0.0
539997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,zero,0,129,0.0
539998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,zero,0,129,0.0


In [17]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders',
       'maternal_disorders_to_recovered_from_maternal_disorders'],
      dtype='str')

In [18]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  3.016016e+06
              2                  1.787592e+06
              3                  2.260803e+06
              4                  2.019465e+06
              5                  1.276479e+06
intervention  1                  3.007554e+06
              2                  1.782590e+06
              3                  2.254428e+06
              4                  2.013768e+06
              5                  1.272610e+06
zero          1                  3.016567e+06
              2                  1.787953e+06
              3                  2.261224e+06
              4                  2.019782e+06
              5                  1.276727e+06
Name: value, dtype: float64

In [19]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [20]:
path = f"{rescaled_child_root}/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if has_simulations:
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"{rescaled_child_root}/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,464.948091
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,368.585793
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,317.995586
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,245.723862
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,252.951034
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,40.953977
23996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,24.090575
23997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,40.953977
23998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,31.317747


In [21]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  200245.674809
              2                  165485.384612
              3                  146680.282021
              4                  139819.286353
              5                  132286.163653
intervention  1                  200168.584971
              2                  165405.885715
              3                  146646.555216
              4                  139771.105204
              5                  132218.710044
zero          1                  200257.720097
              2                  165487.793669
              3                  146682.691078
              4                  139826.513526
              5                  132286.163653
Name: value, dtype: float64

In [22]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [23]:
path = f"{anemia_root}/{vehicle}/{location}/anemia_cases.parquet"
if has_anemia_model:
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"{anemia_root}/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,39533.931348,zero
1,Female,0.0,0.019178,2,32541.983969,zero
2,Female,0.0,0.019178,3,28486.371084,zero
3,Female,0.0,0.019178,4,25082.842833,zero
4,Female,0.0,0.019178,5,18910.719735,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,11273.249048,intervention
746,Male,95.0,125.000000,2,11219.195183,intervention
747,Male,95.0,125.000000,3,11357.917797,intervention
748,Male,95.0,125.000000,4,11553.958149,intervention


In [24]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.305948e+08
              2                  1.227861e+08
              3                  1.207485e+08
              4                  1.148412e+08
              5                  1.072615e+08
intervention  1                  1.296237e+08
              2                  1.218447e+08
              3                  1.197848e+08
              4                  1.138455e+08
              5                  1.061635e+08
zero          1                  1.306546e+08
              2                  1.228442e+08
              3                  1.208079e+08
              4                  1.149025e+08
              5                  1.073292e+08
Name: value, dtype: float64

In [25]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.331612e+08
              2                  1.248051e+08
              3                  1.224653e+08
              4                  1.163536e+08
              5                  1.085113e+08
intervention  1                  1.321732e+08
              2                  1.238497e+08
              3                  1.214885e+08
              4                  1.153455e+08
              5                  1.073998e+08
zero          1                  1.332222e+08
              2                  1.248641e+08
              3                  1.225256e+08
              4                  1.164158e+08
              5                  1.085798e+08
Name: value, dtype: float64

In [26]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [27]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if has_ntd_model:
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  3794.031207
              2                  3351.743559
              3                  3008.977770
              4                  2837.208010
              5                  2416.731876
baseline      1                  3792.235805
              2                  3350.044564
              3                  3007.552674
              4                  2835.973953
              5                  2415.279969
intervention  1                  3769.464158
              2                  3328.507429
              3                  2989.477845
              4                  2820.312312
              5                  2396.902948
Name: value, dtype: float64

In [28]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)